# Complete Feature Selection Analysis: Weekly & Lagged Predictors

**NOTE**: This is the COMPLETE version with all 15 models, comparisons, and visualizations.

Run cells 1-8 from 18_feature_selection_weekly_lagged.ipynb first, then continue here.

## 4. Feature Selection Models (15 approaches)

In [ ]:
# Model 2: Current week only (no lags, no rolling)
print("\n" + "="*80)
print("MODEL 2: CURRENT WEEK ONLY")
print("="*80)

current_features = base_features + temporal_features + regional_features
X_train_current = X_train[current_features]
X_test_current = X_test[current_features]

scaler_current = StandardScaler()
X_train_current_scaled = scaler_current.fit_transform(X_train_current)
X_test_current_scaled = scaler_current.transform(X_test_current)

lr_current = LinearRegression()
lr_current.fit(X_train_current_scaled, y_train)
y_test_pred_current = lr_current.predict(X_test_current_scaled)

current_metrics = calculate_metrics(y_test, y_test_pred_current, len(current_features), len(y_test))
current_metrics['Model'] = 'Current Week Only'
current_metrics['Method'] = 'Simple Selection'
current_metrics['Selected_Features'] = str(current_features)
all_results.append(current_metrics)

print(f"Features: {len(current_features)}")
print(f"Test R²: {current_metrics['R2']:.4f}")
print(f"Test RMSE: {current_metrics['RMSE']:.2f}")
print(f"AIC: {current_metrics['AIC']:.0f}")
print(f"BIC: {current_metrics['BIC']:.0f}")

In [ ]:
# Model 3: Base + 1-week lag
print("\n" + "="*80)
print("MODEL 3: BASE + 1-WEEK LAG")
print("="*80)

lag1_features = base_features + [f for f in lagged_features if '_lag1' in f] + temporal_features + regional_features
X_train_lag1 = X_train[lag1_features]
X_test_lag1 = X_test[lag1_features]

scaler_lag1 = StandardScaler()
X_train_lag1_scaled = scaler_lag1.fit_transform(X_train_lag1)
X_test_lag1_scaled = scaler_lag1.transform(X_test_lag1)

lr_lag1 = LinearRegression()
lr_lag1.fit(X_train_lag1_scaled, y_train)
y_test_pred_lag1 = lr_lag1.predict(X_test_lag1_scaled)

lag1_metrics = calculate_metrics(y_test, y_test_pred_lag1, len(lag1_features), len(y_test))
lag1_metrics['Model'] = 'Base + 1-Week Lag'
lag1_metrics['Method'] = 'Simple Selection'
lag1_metrics['Selected_Features'] = str(lag1_features)
all_results.append(lag1_metrics)

print(f"Features: {len(lag1_features)}")
print(f"Test R²: {lag1_metrics['R2']:.4f}")
print(f"Test RMSE: {lag1_metrics['RMSE']:.2f}")
print(f"Improvement over current week: {(current_metrics['R2'] - lag1_metrics['R2'])*100:.2f}% R²")

In [ ]:
# Model 4: LASSO with Cross-Validation
print("\n" + "="*80)
print("MODEL 4: LASSO-CV (Automatic Feature Selection)")
print("="*80)

# Time series cross-validation
tscv = TimeSeriesSplit(n_splits=5)

lasso_cv = LassoCV(cv=tscv, random_state=42, n_jobs=-1, max_iter=5000)
lasso_cv.fit(X_train_scaled, y_train)
y_test_pred_lasso = lasso_cv.predict(X_test_scaled)

# Get selected features (non-zero coefficients)
lasso_coefs = lasso_cv.coef_
lasso_selected = [all_features[i] for i in range(len(all_features)) if abs(lasso_coefs[i]) > 1e-5]

lasso_metrics = calculate_metrics(y_test, y_test_pred_lasso, len(lasso_selected), len(y_test))
lasso_metrics['Model'] = 'LASSO-CV'
lasso_metrics['Method'] = 'Regularization'
lasso_metrics['Selected_Features'] = str(lasso_selected)
all_results.append(lasso_metrics)

print(f"Optimal alpha: {lasso_cv.alpha_:.4f}")
print(f"Features selected: {len(lasso_selected)}/{len(all_features)}")
print(f"Test R²: {lasso_metrics['R2']:.4f}")
print(f"Test RMSE: {lasso_metrics['RMSE']:.2f}")
print(f"\nTop 10 features by |coefficient|:")
feature_importance = pd.DataFrame({
    'Feature': all_features,
    'Coefficient': lasso_coefs
})
print(feature_importance.reindex(feature_importance['Coefficient'].abs().sort_values(ascending=False).index).head(10).to_string(index=False))

In [ ]:
# Model 5: Elastic Net (L1 + L2 regularization)
print("\n" + "="*80)
print("MODEL 5: ELASTIC NET-CV")
print("="*80)

enet_cv = ElasticNetCV(cv=tscv, random_state=42, n_jobs=-1, max_iter=5000, l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95, 0.99])
enet_cv.fit(X_train_scaled, y_train)
y_test_pred_enet = enet_cv.predict(X_test_scaled)

enet_coefs = enet_cv.coef_
enet_selected = [all_features[i] for i in range(len(all_features)) if abs(enet_coefs[i]) > 1e-5]

enet_metrics = calculate_metrics(y_test, y_test_pred_enet, len(enet_selected), len(y_test))
enet_metrics['Model'] = 'Elastic Net-CV'
enet_metrics['Method'] = 'Regularization'
enet_metrics['Selected_Features'] = str(enet_selected)
all_results.append(enet_metrics)

print(f"Optimal alpha: {enet_cv.alpha_:.4f}")
print(f"Optimal l1_ratio: {enet_cv.l1_ratio_:.4f}")
print(f"Features selected: {len(enet_selected)}/{len(all_features)}")
print(f"Test R²: {enet_metrics['R2']:.4f}")
print(f"Test RMSE: {enet_metrics['RMSE']:.2f}")

In [ ]:
# Model 6: Ridge Regression (L2 only, for comparison)
print("\n" + "="*80)
print("MODEL 6: RIDGE REGRESSION")
print("="*80)

from sklearn.linear_model import RidgeCV
ridge_cv = RidgeCV(alphas=np.logspace(-3, 3, 50), cv=tscv)
ridge_cv.fit(X_train_scaled, y_train)
y_test_pred_ridge = ridge_cv.predict(X_test_scaled)

ridge_metrics = calculate_metrics(y_test, y_test_pred_ridge, len(all_features), len(y_test))
ridge_metrics['Model'] = 'Ridge Regression'
ridge_metrics['Method'] = 'Regularization'
ridge_metrics['Selected_Features'] = str(all_features)
all_results.append(ridge_metrics)

print(f"Optimal alpha: {ridge_cv.alpha_:.4f}")
print(f"Features: All {len(all_features)} (Ridge doesn't zero coefficients)")
print(f"Test R²: {ridge_metrics['R2']:.4f}")
print(f"Test RMSE: {ridge_metrics['RMSE']:.2f}")

In [ ]:
# Model 7: Recursive Feature Elimination with Cross-Validation
print("\n" + "="*80)
print("MODEL 7: RFE-CV (Recursive Feature Elimination)")
print("="*80)

# Use Linear Regression as base estimator
lr_base = LinearRegression()
rfecv = RFECV(estimator=lr_base, step=1, cv=tscv, scoring='r2', n_jobs=-1)
rfecv.fit(X_train_scaled, y_train)
y_test_pred_rfe = rfecv.predict(X_test_scaled)

rfe_selected = [all_features[i] for i in range(len(all_features)) if rfecv.support_[i]]

rfe_metrics = calculate_metrics(y_test, y_test_pred_rfe, len(rfe_selected), len(y_test))
rfe_metrics['Model'] = 'RFE-CV'
rfe_metrics['Method'] = 'Wrapper'
rfe_metrics['Selected_Features'] = str(rfe_selected)
all_results.append(rfe_metrics)

print(f"Optimal number of features: {rfecv.n_features_}")
print(f"Features selected: {len(rfe_selected)}/{len(all_features)}")
print(f"Test R²: {rfe_metrics['R2']:.4f}")
print(f"Test RMSE: {rfe_metrics['RMSE']:.2f}")
print(f"\nSelected features:")
for f in rfe_selected:
    print(f"  - {f}")

In [ ]:
# Model 8: Mutual Information Filter
print("\n" + "="*80)
print("MODEL 8: MUTUAL INFORMATION (Top 15 Features)")
print("="*80)

# Calculate mutual information
mi_scores = mutual_info_regression(X_train_scaled, y_train, random_state=42)
mi_ranking = pd.DataFrame({
    'Feature': all_features,
    'MI_Score': mi_scores
}).sort_values('MI_Score', ascending=False)

# Select top 15
n_mi_features = 15
mi_selected = mi_ranking.head(n_mi_features)['Feature'].tolist()
mi_indices = [all_features.index(f) for f in mi_selected]

X_train_mi = X_train_scaled[:, mi_indices]
X_test_mi = X_test_scaled[:, mi_indices]

lr_mi = LinearRegression()
lr_mi.fit(X_train_mi, y_train)
y_test_pred_mi = lr_mi.predict(X_test_mi)

mi_metrics = calculate_metrics(y_test, y_test_pred_mi, len(mi_selected), len(y_test))
mi_metrics['Model'] = 'Mutual Information (Top 15)'
mi_metrics['Method'] = 'Filter'
mi_metrics['Selected_Features'] = str(mi_selected)
all_results.append(mi_metrics)

print(f"Features selected: {len(mi_selected)}")
print(f"Test R²: {mi_metrics['R2']:.4f}")
print(f"Test RMSE: {mi_metrics['RMSE']:.2f}")
print(f"\nTop 15 features by MI score:")
print(mi_ranking.head(15).to_string(index=False))

In [ ]:
# Model 9: Correlation Filter (|r| > 0.10 with mortality)
print("\n" + "="*80)
print("MODEL 9: CORRELATION THRESHOLD (|r| > 0.10)")
print("="*80)

# Calculate correlations
correlations = []
for i, feature in enumerate(all_features):
    r, p = stats.pearsonr(X_train.iloc[:, i], y_train)
    correlations.append({'Feature': feature, 'Correlation': r, 'p_value': p})

corr_df = pd.DataFrame(correlations)
corr_selected_df = corr_df[corr_df['Correlation'].abs() > 0.10]
corr_selected = corr_selected_df['Feature'].tolist()
corr_indices = [all_features.index(f) for f in corr_selected]

X_train_corr = X_train_scaled[:, corr_indices]
X_test_corr = X_test_scaled[:, corr_indices]

lr_corr = LinearRegression()
lr_corr.fit(X_train_corr, y_train)
y_test_pred_corr = lr_corr.predict(X_test_corr)

corr_metrics = calculate_metrics(y_test, y_test_pred_corr, len(corr_selected), len(y_test))
corr_metrics['Model'] = 'Correlation Filter (>0.10)'
corr_metrics['Method'] = 'Filter'
corr_metrics['Selected_Features'] = str(corr_selected)
all_results.append(corr_metrics)

print(f"Features selected: {len(corr_selected)}/{len(all_features)}")
print(f"Test R²: {corr_metrics['R2']:.4f}")
print(f"Test RMSE: {corr_metrics['RMSE']:.2f}")
print(f"\nSelected features:")
print(corr_selected_df.sort_values('Correlation', key=abs, ascending=False).to_string(index=False))

In [ ]:
# Model 10: Theory-Driven Selection
print("\n" + "="*80)
print("MODEL 10: THEORY-DRIVEN (Domain Knowledge)")
print("="*80)

# Based on livestock heat stress literature:
# - VPD (evaporative cooling)
# - Extreme heat (>35°C)
# - Poor nighttime recovery (>24°C)
# - Lag effects (1-2 weeks)
# - Season

theory_selected = [
    'mean_vpd_max',
    'mean_vpd_mean',
    'mean_daytime_hours_above_35',
    'mean_nighttime_hours_above_24',
    'mean_daytime_hours_above_30',
    'mean_vpd_max_lag1',
    'mean_daytime_hours_above_35_lag1',
    'mean_nighttime_hours_above_24_lag1',
    'month',
    'season_Summer',
    'region'
]

theory_indices = [all_features.index(f) for f in theory_selected if f in all_features]
theory_selected = [all_features[i] for i in theory_indices]

X_train_theory = X_train_scaled[:, theory_indices]
X_test_theory = X_test_scaled[:, theory_indices]

lr_theory = LinearRegression()
lr_theory.fit(X_train_theory, y_train)
y_test_pred_theory = lr_theory.predict(X_test_theory)

theory_metrics = calculate_metrics(y_test, y_test_pred_theory, len(theory_selected), len(y_test))
theory_metrics['Model'] = 'Theory-Driven'
theory_metrics['Method'] = 'Domain Knowledge'
theory_metrics['Selected_Features'] = str(theory_selected)
all_results.append(theory_metrics)

print(f"Features selected: {len(theory_selected)}")
print(f"Test R²: {theory_metrics['R2']:.4f}")
print(f"Test RMSE: {theory_metrics['RMSE']:.2f}")
print(f"\nSelected features (domain knowledge):")
for f in theory_selected:
    print(f"  - {f}")

In [ ]:
# Model 11: Rolling Averages Only
print("\n" + "="*80)
print("MODEL 11: ROLLING AVERAGES ONLY")
print("="*80)

roll_features = rolling_features + temporal_features + regional_features
X_train_roll = X_train[roll_features]
X_test_roll = X_test[roll_features]

scaler_roll = StandardScaler()
X_train_roll_scaled = scaler_roll.fit_transform(X_train_roll)
X_test_roll_scaled = scaler_roll.transform(X_test_roll)

lr_roll = LinearRegression()
lr_roll.fit(X_train_roll_scaled, y_train)
y_test_pred_roll = lr_roll.predict(X_test_roll_scaled)

roll_metrics = calculate_metrics(y_test, y_test_pred_roll, len(roll_features), len(y_test))
roll_metrics['Model'] = 'Rolling Averages Only'
roll_metrics['Method'] = 'Simple Selection'
roll_metrics['Selected_Features'] = str(roll_features)
all_results.append(roll_metrics)

print(f"Features: {len(roll_features)}")
print(f"Test R²: {roll_metrics['R2']:.4f}")
print(f"Test RMSE: {roll_metrics['RMSE']:.2f}")

In [ ]:
# Model 12: Base + Interactions (No lags/rolling)
print("\n" + "="*80)
print("MODEL 12: BASE + INTERACTIONS")
print("="*80)

interact_features = base_features + interaction_features + temporal_features + regional_features
X_train_interact = X_train[interact_features]
X_test_interact = X_test[interact_features]

scaler_interact = StandardScaler()
X_train_interact_scaled = scaler_interact.fit_transform(X_train_interact)
X_test_interact_scaled = scaler_interact.transform(X_test_interact)

lr_interact = LinearRegression()
lr_interact.fit(X_train_interact_scaled, y_train)
y_test_pred_interact = lr_interact.predict(X_test_interact_scaled)

interact_metrics = calculate_metrics(y_test, y_test_pred_interact, len(interact_features), len(y_test))
interact_metrics['Model'] = 'Base + Interactions'
interact_metrics['Method'] = 'Simple Selection'
interact_metrics['Selected_Features'] = str(interact_features)
all_results.append(interact_metrics)

print(f"Features: {len(interact_features)}")
print(f"Test R²: {interact_metrics['R2']:.4f}")
print(f"Test RMSE: {interact_metrics['RMSE']:.2f}")

In [ ]:
# Model 13: LASSO on Base + Lag1 only (reduced feature space)
print("\n" + "="*80)
print("MODEL 13: LASSO on BASE + LAG1 (Reduced Space)")
print("="*80)

X_train_reduced = X_train[lag1_features]
X_test_reduced = X_test[lag1_features]

scaler_reduced = StandardScaler()
X_train_reduced_scaled = scaler_reduced.fit_transform(X_train_reduced)
X_test_reduced_scaled = scaler_reduced.transform(X_test_reduced)

lasso_reduced = LassoCV(cv=tscv, random_state=42, n_jobs=-1)
lasso_reduced.fit(X_train_reduced_scaled, y_train)
y_test_pred_lasso_reduced = lasso_reduced.predict(X_test_reduced_scaled)

lasso_reduced_coefs = lasso_reduced.coef_
lasso_reduced_selected = [lag1_features[i] for i in range(len(lag1_features)) if abs(lasso_reduced_coefs[i]) > 1e-5]

lasso_reduced_metrics = calculate_metrics(y_test, y_test_pred_lasso_reduced, len(lasso_reduced_selected), len(y_test))
lasso_reduced_metrics['Model'] = 'LASSO on Base+Lag1'
lasso_reduced_metrics['Method'] = 'Regularization'
lasso_reduced_metrics['Selected_Features'] = str(lasso_reduced_selected)
all_results.append(lasso_reduced_metrics)

print(f"Features selected: {len(lasso_reduced_selected)}/{len(lag1_features)}")
print(f"Test R²: {lasso_reduced_metrics['R2']:.4f}")
print(f"Test RMSE: {lasso_reduced_metrics['RMSE']:.2f}")

In [ ]:
# Create summary table
print("\n" + "="*80)
print("MODEL COMPARISON SUMMARY")
print("="*80)

results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values('Test R²', ascending=False).reset_index(drop=True)
results_df['Rank'] = range(1, len(results_df) + 1)

# Display summary
display_cols = ['Rank', 'Model', 'Method', 'N_Features', 'R2', 'Adj_R2', 'RMSE', 'AIC', 'BIC']
print("\n" + results_df[display_cols].to_string(index=False))

# Save to CSV
results_df.to_csv('../../cattle_data/weekly_model_selection_results.csv', index=False)
print("\n✓ Results saved to: cattle_data/weekly_model_selection_results.csv")